In [1]:
using Revise
includet("../../scripts/single_influx.jl")

In [2]:
using ProgressMeter
using ColorSchemes
using UnPack

In [3]:
includet("../../scripts/figures_util.jl")

using GLMakie
using CairoMakie
CairoMakie.activate!()

# Prepping data for PDE run v1
This does ~10 values for li (overkill tbf...) and for each it does 5 values of K accross the unstable region. For each of those it does the first 20 of the randomly generated, solved systems

In [ ]:
f = jldopen("../../cluster_env/runs/single_influx/gd5_forfigures_260212_153215.jld2");
Ks, lis, results = make_Kli_matrix_raw(f);

dominant_outcomes = map(results) do d
    mode([r.linstab_outcome for r in d])
end;

length(Ks), length(lis), size(results)

Progress:   5%|██▏                                      |  ETA: 0:00:09

In [145]:
Kis_to_sample = []

lis_to_sample = 20:length(lis)

for i in lis_to_sample
    first_unstable = nothing
    last_unstable = nothing
    for j in 1:length(Ks)
        if isnothing(first_unstable) && (dominant_outcomes[j, i] == :unstable)
            first_unstable = j
        end
        if isnothing(last_unstable) && (dominant_outcomes[j, i] == :stable)
            last_unstable = j - 1
        end
    end

    step = round(Int, (last_unstable - first_unstable) / 4)
    push!(Kis_to_sample, [first_unstable, first_unstable + step, first_unstable + 2 * step, first_unstable + 3 * step, last_unstable])

#=     if isnothing(first_unstable)
        push!(Kis_to_sample, [])
    else
        diff = last_unstable - first_unstable
        if diff <= 3
            push!(Kis_to_sample, [round(Int, (first_unstable + last_unstable) / 2)])
        elseif diff <= 6
            step = round(Int, (last_unstable - first_unstable) / 2)
            push!(Kis_to_sample, [first_unstable, first_unstable + step, last_unstable])
        else
            push!(Kis_to_sample, nothing)
        end
    end =#
end

In [152]:
systems_to_run = []

for mi in 1:length(lis_to_sample)
    for K_i in Kis_to_sample[mi]
        li_i = lis_to_sample[mi]
        
        # systems = results[K_i, li_i]
        systems = results[K_i, li_i][1:20] # only do 20 at every K, li
        
        push!(systems_to_run, (;
            K=Ks[K_i], li=lis[li_i],
            systems
        ))
    end
end

total_runs = sum(systems_to_run) do x length(x.systems) end
@show total_runs;

total_runs = 1100


In [154]:
jldsave("../../cluster_env/runs/single_influx_pdes2/systems1.jld2"; systems_to_run)